# Domain adaptation

Implementation of various UDA techniques on BDAPPV. 

In [ ]:
# Libraries
import sys
sys.path.append('../')

import os
import pandas as pd
import numpy as np
import json
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt
from src import bdappv, utils, helpers
import torchvision
from torch.utils.data import DataLoader
import tqdm
import torch
import random
from functools import reduce 
import copy
from spectral_sobol.torch_explainer import WaveletSobol
from torch.nn import functional as F
import os
import pywt
from src import reconstruction
import cv2
from torchvision.models import resnet50
import torch.nn as nn

## Dataset construction 

We focus on the varying acquisition conditions and set up a test dataset that contains iamges of the same PV panels on IGN and Google images and a training dataset that contains labelled images from IGN and Google (independent of the test dataset). 

In [2]:
# Data preparation: train and test instances
# focus on the sensitivty to acquisition conditoins

images_list = json.load(open("../data/images_lists.json"))

dataset_dir = "../../../data/bdappv"
batch_size = 512

# baseline transforms: no corruptions
baseline = torchvision.transforms.Compose([
    torchvision.transforms.ToPILImage(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean = (0.485, 0.456, 0.406), std = (0.229, 0.224, 0.225)),
])

datasets = {}

for case in ['google', 'ign']:

    # test instances: test source and test target

    path = os.path.join(dataset_dir, case)

    if case == "google":

        dataset = bdappv.BDAPPVClassification(path, size = 200, transform=baseline, images_list=images_list["test"], random = False, downsample=200)
    else:
        dataset = bdappv.BDAPPVClassification(path, size = 200, images_list=images_list["test"], random = False, transform = baseline)

    database = DataLoader(dataset, batch_size=batch_size)
    datasets[case] = database

    # train instances: Google train/IGN train
    case_name='train_{}'.format(case)
    if case=="google":
        dataset = bdappv.BDAPPVClassification(path, size = 200, transform=baseline, images_list=images_list["train"], random = False, downsample=200)
    else:
        dataset = bdappv.BDAPPVClassification(path, size = 200, images_list=images_list["train"], random = False, transform = baseline)

    database=DataLoader(dataset,batch_size=batch_size)
    datasets[case_name]=database



In [ ]:
# visualization

test_google=next(iter(datasets['google']))
test_ign=next(iter(datasets['ign']))

train_google=next(iter(datasets['train_google']))
train_ign=next(iter(datasets['train_ign']))

fig, ax = plt.subplots(4,4, figsize=(16,16))

def reshape_and_normalize(tensor):
    """
    Reshape a (3, H, W) tensor to (H, W, 3) and normalize values between 0 and 1.
    
    Parameters:
        tensor (numpy.ndarray): Input tensor with shape (3, H, W).
    
    Returns:
        numpy.ndarray: Reshaped and normalized array with shape (H, W, 3).
    """
    # Check input dimensions
    if tensor.shape[0] != 3:
        raise ValueError("Input tensor must have shape (3, H, W)")
    
    # Reshape to (H, W, 3)
    reshaped_array = np.transpose(tensor.numpy(), (1, 2, 0))
    
    # Normalize to [0, 1]
    normalized_array = (reshaped_array - np.min(reshaped_array)) / (np.max(reshaped_array)-np.min(reshaped_array))
    
    return normalized_array

for i, (instance, name) in enumerate(zip(
    [test_google, test_ign, train_google, train_ign], ['Google test', 'IGN test', 'Google train', 'IGN train']
)):
    
    # pick random indices

    images=instance[0]
    ax[0,i].set_title(name)

    ax[0,i].imshow(reshape_and_normalize(images[0]))
    ax[1,i].imshow(reshape_and_normalize(images[-1]))
    ax[2,i].imshow(reshape_and_normalize(images[42]))
    ax[3,i].imshow(reshape_and_normalize(images[95]))

    ax[0,i].axis('off')
    ax[1,i].axis('off')
    ax[2,i].axis('off')
    ax[3,i].axis('off')


plt.show()

## DA methods - quantitative evaluation

### DeepCoral

In [ ]:
da_models={}

device="cuda"

CORAL_DIR='../deepcoral'
coral_name="model_weights.pth"

model=resnet50(pretrained=False)
model.fc = nn.Linear(2048, 19589) 
weights=torch.load('../weights-da/coral.pth') # coral model weights

model.load_state_dict(weights)
model.to(device)
model.eval()
threshold=0.5

da_models['coral']=model

results={}

#for case in ['ign', 'google']: # test on the target and source dataset
    #print(case)
    #results[case]=utils.evaluate(model,datasets[case],device,threshold)

In [ ]:
# table to display the results with the F1 score and the rates

header = "Case & F1 Score & True positive rate & True negative rate & False positive rate & False negative rate"
print(header)
for case in ['google', 'ign']:

    f1=results[case][0]
    # tp
    tp=len(results[case][5]['tp'])
    tn=len(results[case][5]['tn'])
    fp=len(results[case][5]['fp'])
    fn=len(results[case][5]['fn'])

    positives=tp+fn
    negatives=tn+fp

    line= '{} & {:0.2f} & {:0.2f} & {:0.2f} & {:0.2f} & {:0.2f}'.format(
        case, f1, tp / positives, tn / negatives, fp / negatives, fn / positives
    )

    print(line)

### ADDA, WGGRL, and RevGrad

These three methods were implemented using the same library. We provide the code to evaluate these methods on our datasets.

In [6]:
# Custom model class

import torch
import torch.nn as nn
from torchvision.models import resnet50

class Net(nn.Module):
    """
    custom model class that is used in the Git repository https://github.com/jvanvugt/pytorch-domain-adaptation/tree/master

    this model features a feature_extractor that is used to thrain the modes. We have replaced this feature
    extractor by a ResNet-50 to match the architecture used throughout our paper.
    """
    def __init__(self):
        super().__init__()
        # Load ResNet-50 as the feature extractor
        resnet = resnet50(pretrained=True)
        
        # Remove the fully connected layer of ResNet-50
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])  # Output: [batch_size, 2048, 1, 1]
        
        # Define the classifier for two output classes
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),  # Map ResNet-50's output to a smaller dimension
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)  # Two output classes
        )

    def forward(self, x):
        # Extract features using ResNet-50
        features = self.feature_extractor(x)
        features = torch.flatten(features, 1)  # Flatten the output: [batch_size, 2048]
        
        # Pass the features through the classifier
        logits = self.classifier(features)
        return logits


In [ ]:
device="cuda:1"
DA_DIR='../weights-da'

methods_names=['adda', "wdgrl", 'revgrad']

models={}

for method in methods_names:
    weights=torch.load(os.path.join(DA_DIR, method+'.pt'))
    model=Net().to(device)
    model.load_state_dict(weights)
    model.eval()

    models[method]=model

    da_models[method]=model

print('Models loaded.')

results={method : {} for method in methods_names}
threshold=0.5

# for method in methods_names:
# 
#     model=models[method]
#     print('Evaluating method .... {}'.format(method))
#     
#     for case in ['ign', 'google']: # test on the target and source dataset
# 
#         print('Inference on the dataset .... {}'.format(case))
#         results[method][case]=utils.evaluate(model,datasets[case],device,threshold)

In [ ]:
# table to display the results with the F1 score and the rates

header = "Case & F1 Score & True positive rate & True negative rate & False positive rate & False negative rate"
print(header)

for method in methods_names:
    method_header='.............................................{}...........................................'.format(method)
    print(method_header)

    for case in ['google', 'ign']:

        f1, tp, fp, tn, fn, _, _, _=results[method][case]

        positives=tp+fn
        negatives=tn+fp

        line= '{} & {:0.2f} & {:0.2f} & {:0.2f} & {:0.2f} & {:0.2f}'.format(
            case, f1, tp / positives, tn / negatives, fp / negatives, fn / positives
        )

        print(line)

### Visualization of the predictions with the WCAM

This code generates the figure presented in the appendix that analyze the UDA model's decisions with the WCAM.

In [30]:
# helpers

def rgb_to_wavelet_array(image, wavelet='haar', level=3):
    # Convert PIL image to NumPy array
    img_array = np.array(image.convert('L'))

    # Compute wavelet transform for each channel
    c = pywt.wavedec2(img_array, wavelet, level=level)     
    # normalize each coefficient array independently for better visibility
    c[0] /= np.abs(c[0]).max()
    for detail_level in range(level):
        c[detail_level + 1] = [d/np.abs(d).max() for d in c[detail_level + 1]]
    arr, _ = pywt.coeffs_to_array(c)

    
    return arr


def plot_wcam(ax, image, wcam, levels, vmin = None, vmax = None):
    """
    plts the wcam
    """

    def logplot(x):
        return np.log(1 + x)
    
    size = image.size[0]
    # compute the wavelet transform
    wt = rgb_to_wavelet_array(image,level = levels)
    
    # plots
    ax.imshow(wt, cmap = 'gray')

    vmin = logplot(vmin) if vmin is not None else vmin
    vmax = logplot(vmax) if vmax is not None else vmax

    #im = ax.imshow(1 + logplot(wcam), cmap = "hot", alpha = 0.5, vmin = vmin, vmax = vmax)

    minlog = np.min(logplot(wcam))
    im = ax.imshow(minlog  + logplot(wcam), cmap = "hot", alpha = 0.7, vmin = vmin, vmax = vmax)

    ax.axis('off')
    utils.add_lines(size, levels, ax)

    #cbar = plt.colorbar(im, ax = ax)
    #cbar.ax.tick_params(labelsize=10)

    return None

In [ ]:
grid_size=40
nb_design=4
device='cuda'

wcams = {}

# images
x_google, y_google, _=test_google
x_ign, y_ign,_=test_ign

n_samples=8
np.random.seed(42)
num_wcam=x_google.shape[0]
sample_indices=np.random.choice(n_samples,num_wcam)

for model_name in da_models.keys():

    print('Computing the WCAM for the model .......... {}'.format(model_name))

    model=da_models[model_name]
    model.to(device)

    wcams[model_name]={}
    

    # pick the desired model 
    wavelet = WaveletSobol(model, grid_size = grid_size, nb_design= nb_design, \
                        batch_size = 128, opt = {'size' : grid_size, "approximation": True})


    wcams_ign=wavelet(x_ign[:n_samples],y_ign[:n_samples])
    wcams_google=wavelet(x_google[:n_samples],y_google[:n_samples])

    wcams[model_name]['google']=wcams_google
    wcams[model_name]['ign']=wcams_ign

In [ ]:
from PIL import Image

fig, ax=plt.subplots(4,4, figsize=(16,16))

method_names=['DeepCORAL', "ADDA", "WDGRL", "RevGrad"]

# each column is a method

for (col, method_name), key in zip(enumerate(method_names), list(wcams.keys())):

    # titles
    ax[0,col].set_title(method_name + '\n Image (Google)') 
    ax[1,col].set_title('WCAM') 
    ax[2,col].set_title('Image (IGN)') 
    ax[3,col].set_title('WCAM')

    index=8

    # images
    img_ign=reshape_and_normalize(x_ign[index].squeeze(0))
    img_google=reshape_and_normalize(x_google[index].squeeze(0))

    img_google=Image.fromarray((img_google * 255).astype(np.uint8))
    img_ign=Image.fromarray((img_ign * 255).astype(np.uint8))

    ax[2,col].imshow(img_ign)
    ax[0,col].imshow(img_google)

    plot_wcam(ax[1,col], img_google, 
              cv2.resize(wcams[key]['google'][index], (200,200), interpolation=cv2.INTER_LINEAR),3)

    plot_wcam(ax[3,col], img_ign, 
              cv2.resize(wcams[key]['ign'][index], (200,200), interpolation=cv2.INTER_LINEAR),3)

    ax[0,col].axis('off') 
    ax[1,col].axis('off') 
    ax[2,col].axis('off') 
    ax[3,col].axis('off') 

# plt.savefig('wcam-da.pdf', bbox_inches='tight')
plt.show()